In [ ]:
# ==========================================================
# ACTIVIDAD 1. Inventario del repositorio
# No descarga los precios ni entrena modelos.
# ==========================================================
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from google.colab import drive
from huggingface_hub import HfApi, hf_hub_download

# ----------------------------------------------------------
# 1. Configuración
# ----------------------------------------------------------
REPO_ID = "AmirTrader/YahooFinance"
COMMIT = "4c921233b7cc7b36c56891d21bed8c4f5010d7be"  # versión congelada

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
CARPETA.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# ----------------------------------------------------------
# 2. Verificar la versión congelada
# ----------------------------------------------------------
api = HfApi()
info_congelada = api.dataset_info(REPO_ID, revision=COMMIT)
info_actual = api.dataset_info(REPO_ID)

print("=== VERSIÓN ===")
print("Commit congelado:", info_congelada.sha)
print("Fecha del commit:", info_congelada.last_modified)
print("Commit más reciente hoy:", info_actual.sha)

# ----------------------------------------------------------
# 3. Listar los archivos de data/daily, sin descargarlos
# ----------------------------------------------------------
elementos = api.list_repo_tree(
    repo_id=REPO_ID,
    repo_type="dataset",
    path_in_repo="data/daily",
    revision=COMMIT,
    recursive=False,
)

filas = []
otros_elementos = []

for elemento in elementos:
    nombre = Path(elemento.path).name
    if nombre.endswith(".parquet"):
        filas.append({
            "ticker": nombre.removesuffix(".parquet"),
            "archivo": nombre,
            "tamano_kb": round(elemento.size / 1000, 2),
        })
    else:
        otros_elementos.append(nombre)

inventario = pd.DataFrame(filas).sort_values("ticker").reset_index(drop=True)

print("\n=== INVENTARIO DE data/daily ===")
print("Archivos parquet:", len(inventario))
print("Otros elementos:", len(otros_elementos), otros_elementos[:10])
print("Tamaño total (MB):", round(inventario["tamano_kb"].sum() / 1000, 1))
print("Tickers repetidos:", inventario["ticker"].str.upper().duplicated().sum())
print("\nTamaño por archivo (kB):")
print(inventario["tamano_kb"].describe().round(2))
print("\nPrimeros 10 tickers:", inventario["ticker"].head(10).tolist())
print("Últimos 10 tickers:", inventario["ticker"].tail(10).tolist())

# ----------------------------------------------------------
# 4. Examinar el archivo de estado
# ----------------------------------------------------------
ruta_estado = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="data/state/ingestion_state.parquet",
    revision=COMMIT,
)
estado = pd.read_parquet(ruta_estado)

print("\n=== ARCHIVO DE ESTADO ===")
print("Filas y columnas:", estado.shape)
print("\nColumnas y tipos:")
print(estado.dtypes)
print("\nValores nulos por columna:")
print(estado.isna().sum())
print("\nPrimeras 5 filas:")
print(estado.head(5))

# Comparar el estado con los archivos, si existe una columna identificadora
columna_id = next(
    (c for c in estado.columns if c.lower() in ("ticker", "symbol")), None
)
if columna_id is None:
    print("\nNo hay una columna llamada ticker o symbol: la revisaremos juntos.")
else:
    en_estado = set(estado[columna_id].astype(str))
    en_archivos = set(inventario["ticker"])
    print(f"\nComparación usando la columna '{columna_id}':")
    print("En ambos:", len(en_estado & en_archivos))
    print("Solo en el estado:", len(en_estado - en_archivos))
    print("Solo en los archivos:", len(en_archivos - en_estado))

# ----------------------------------------------------------
# 5. Guardar el inventario y el registro de versión
# ----------------------------------------------------------
inventario.to_csv(CARPETA / "inventario_archivos.csv", index=False)

registro = {
    "repositorio": REPO_ID,
    "commit": COMMIT,
    "fecha_commit": str(info_congelada.last_modified),
    "fecha_consulta_utc": datetime.now(timezone.utc).isoformat(),
    "commit_mas_reciente_al_consultar": info_actual.sha,
    "archivos_parquet": len(inventario),
    "tamano_total_mb": round(inventario["tamano_kb"].sum() / 1000, 1),
    "filas_estado": len(estado),
    "columnas_estado": list(estado.columns),
}
with open(CARPETA / "registro_version.json", "w", encoding="utf-8") as archivo:
    json.dump(registro, archivo, ensure_ascii=False, indent=2)

print("\n=== ARCHIVOS GUARDADOS ===")
for ruta in sorted(CARPETA.iterdir()):
    print(ruta.name, "-", ruta.stat().st_size, "bytes")

Mounted at /content/drive
=== VERSIÓN ===
Commit congelado: 4c921233b7cc7b36c56891d21bed8c4f5010d7be
Fecha del commit: 2026-11-04 07:18:07+00:00
Commit más reciente hoy: 6a1c45437549064f95b7694cd6998bb19f5e66a1

=== INVENTARIO DE data/daily ===
Archivos parquet: 6818
Otros elementos: 0 []
Tamaño total (MB): 362.7
Tickers repetidos: 0

Tamaño por archivo (kB):
count    6818.00
mean       53.20
std        24.25
min         4.43
25%        39.01
50%        57.84
75%        73.16
max        90.06
Name: tamano_kb, dtype: float64

Primeros 10 tickers: ['A', 'AA', 'AAC', 'AACG', 'AACI', 'AACIU', 'AACO', 'AACOU', 'AACP', 'AACPU']
Últimos 10 tickers: ['ZUMZ', 'ZURA', 'ZVIA', 'ZVRA', 'ZWS', 'ZYBT', 'ZYME', 'ZZZTA', 'ZZZTS', 'ZZZTX']


data/state/ingestion_state.parquet: reconstructing file:   0%|          |  0.00B / 63.5kB            

data/state/ingestion_state.parquet: downloading bytes:           |  0.00B            


=== ARCHIVO DE ESTADO ===
Filas y columnas: (6871, 7)

Columnas y tipos:
ticker            object
last_date         object
status            object
last_attempt      object
last_success      object
last_error        object
failure_count    float64
dtype: object

Valores nulos por columna:
ticker            0
last_date        53
status            0
last_attempt      0
last_success     53
last_error        0
failure_count    53
dtype: int64

Primeras 5 filas:
  ticker   last_date   status                      last_attempt                      last_success                              last_error  failure_count
0      A  2026-11-03  success  2026-11-04T07:04:19.818400+00:00  2026-11-04T07:04:19.818400+00:00                                                    0.0
1     AA  2026-11-03  success  2026-11-04T07:04:19.818400+00:00  2026-11-04T07:04:19.818400+00:00                                                    0.0
2    AAC  2026-11-03  success  2026-11-04T07:04:19.818400+00:00  2026-11-04T07

In [ ]:
# ==========================================================
# ACTIVIDAD 2. Diagnóstico del archivo de estado
# Usa el mismo commit congelado. No descarga precios.
# ==========================================================
from pathlib import Path

import pandas as pd
from google.colab import drive
from huggingface_hub import hf_hub_download

REPO_ID = "AmirTrader/YahooFinance"
COMMIT = "4c921233b7cc7b36c56891d21bed8c4f5010d7be"
FIN_PERIODO = pd.Timestamp("2026-08-31")       # fin provisional del estudio
ULTIMA_SESION = pd.Timestamp("2026-11-03")     # última fecha esperada en este commit

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# ----------------------------------------------------------
# 1. Cargar el inventario de la Actividad 1 y el archivo de estado
# ----------------------------------------------------------
# keep_default_na=False evita que tickers como "NA" se lean como vacíos
inventario = pd.read_csv(
    CARPETA / "inventario_archivos.csv",
    dtype={"ticker": str},
    keep_default_na=False,
)
ruta_estado = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="data/state/ingestion_state.parquet",
    revision=COMMIT,
)
estado = pd.read_parquet(ruta_estado)

print("=== CONTROL DE CARGA ===")
print("Inventario:", len(inventario), "filas |", inventario["ticker"].nunique(), "tickers únicos")
print("Estado:", len(estado), "filas |", estado["ticker"].nunique(), "tickers únicos")
print("Ejecuciones distintas en last_attempt:", estado["last_attempt"].nunique())

# ----------------------------------------------------------
# 2. Relación entre el estado de descarga y la existencia del archivo
# ----------------------------------------------------------
estado["tiene_archivo"] = estado["ticker"].isin(inventario["ticker"])

print("\n=== STATUS vs. EXISTENCIA DE ARCHIVO ===")
print(pd.crosstab(estado["status"], estado["tiene_archivo"],
                  margins=True, margins_name="Total"))

# ----------------------------------------------------------
# 3. Errores: un texto vacío no cuenta como nulo, así que lo limpiamos
# ----------------------------------------------------------
error_limpio = estado["last_error"].fillna("").str.strip()

print("\n=== ERRORES ===")
print("Registros con texto de error:", (error_limpio != "").sum())
print("\nMensajes más frecuentes:")
print(error_limpio[error_limpio != ""].value_counts().head(10))
print("\nIntentos fallidos (failure_count):")
print(estado["failure_count"].value_counts(dropna=False).sort_index())

# ----------------------------------------------------------
# 4. Última fecha disponible de los tickers que sí tienen archivo
# ----------------------------------------------------------
estado["last_date"] = pd.to_datetime(estado["last_date"], errors="coerce")
con_archivo = estado[estado["tiene_archivo"]]

print("\n=== ÚLTIMA FECHA (tickers con archivo) ===")
print("Mínima:", con_archivo["last_date"].min())
print("Máxima:", con_archivo["last_date"].max())
print("Sin fecha:", con_archivo["last_date"].isna().sum())
print("Terminan antes del 31-ago-2026:", (con_archivo["last_date"] < FIN_PERIODO).sum())
print("Terminan antes del 3-nov-2026:", (con_archivo["last_date"] < ULTIMA_SESION).sum())

print("\nFechas finales más frecuentes:")
print(con_archivo["last_date"].dt.date.value_counts().head(10))

print("\nTickers con archivo que terminan antes del 31-ago-2026 (máx. 30):")
print(
    con_archivo.loc[con_archivo["last_date"] < FIN_PERIODO,
                    ["ticker", "last_date", "status", "last_error"]]
    .sort_values("last_date")
    .head(30)
    .to_string(index=False)
)

# ----------------------------------------------------------
# 5. Tickers registrados en el estado que no tienen archivo
# ----------------------------------------------------------
sin_archivo = sorted(estado.loc[~estado["tiene_archivo"], "ticker"])
print("\n=== TICKERS SIN ARCHIVO:", len(sin_archivo), "===")
print(sin_archivo)

# ----------------------------------------------------------
# 6. Guardar la tabla de control
# ----------------------------------------------------------
estado.to_csv(CARPETA / "estado_ingesta.csv", index=False)
print("\nGuardado: estado_ingesta.csv -", (CARPETA / "estado_ingesta.csv").stat().st_size, "bytes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== CONTROL DE CARGA ===
Inventario: 6818 filas | 6818 tickers únicos
Estado: 6871 filas | 6871 tickers únicos
Ejecuciones distintas en last_attempt: 249

=== STATUS vs. EXISTENCIA DE ARCHIVO ===
tiene_archivo  False  True  Total
status                           
empty             53     0     53
success            0  6818   6818
Total             53  6818   6871

=== ERRORES ===
Registros con texto de error: 53

Mensajes más frecuentes:
last_error
Parsed data was empty after validation    53
Name: count, dtype: int64

Intentos fallidos (failure_count):
failure_count
0.0    6818
NaN      53
Name: count, dtype: int64

=== ÚLTIMA FECHA (tickers con archivo) ===
Mínima: 2026-08-20 00:00:00
Máxima: 2026-11-03 00:00:00
Sin fecha: 0
Terminan antes del 31-ago-2026: 1
Terminan antes del 3-nov-2026: 509

Fechas finales más frecuentes:
last_date
2026-11-03    6309
2026

In [3]:
# ==========================================================
# ACTIVIDAD 3. Cobertura temporal y esquema de cada archivo
# Descarga los archivos del commit congelado (~363 MB) al disco
# temporal de Colab. En Drive solo se guarda la tabla resumen.
# ==========================================================
import time
from pathlib import Path

import huggingface_hub
import pandas as pd
from google.colab import drive
from huggingface_hub import snapshot_download

REPO_ID = "AmirTrader/YahooFinance"
COMMIT = "4c921233b7cc7b36c56891d21bed8c4f5010d7be"
INICIO_PERIODO = pd.Timestamp("2019-09-01")
FIN_PERIODO = pd.Timestamp("2026-08-31")
COLUMNAS_ESPERADAS = ["date", "open", "high", "low", "close", "adj_close", "volume"]

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
LOCAL = Path("/content/datos_yahoo")    # disco temporal: se borra al cerrar la sesión

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)

print("Versión de huggingface_hub:", huggingface_hub.__version__)

# ----------------------------------------------------------
# 1. Descargar los archivos de data/daily del commit congelado
# ----------------------------------------------------------
# Si se alcanza el límite de descargas (error 429), espera y continúa.
# Los archivos ya descargados no se vuelven a descargar.
for intento in range(1, 6):
    try:
        snapshot_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            revision=COMMIT,
            allow_patterns=["data/daily/*.parquet"],
            local_dir=LOCAL,
            max_workers=8,
        )
        break
    except Exception as error:
        if "429" not in str(error) or intento == 5:
            raise
        print(f"Límite de descargas alcanzado (intento {intento}). Espero 5 minutos...")
        time.sleep(300)

archivos = sorted((LOCAL / "data" / "daily").glob("*.parquet"))
print("\nArchivos descargados:", len(archivos))

# ----------------------------------------------------------
# 2. Leer cada archivo y registrar su esquema y cobertura
# ----------------------------------------------------------
filas = []
for numero, ruta in enumerate(archivos, start=1):
    df = pd.read_parquet(ruta)

    if "date" in df.columns:
        fechas = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
    else:
        fechas = pd.Series(pd.NaT, index=df.index)

    en_periodo = fechas.between(INICIO_PERIODO, FIN_PERIODO)

    filas.append({
        "ticker": ruta.name.removesuffix(".parquet"),
        "columnas": "|".join(df.columns),
        "tipos": "|".join(df.dtypes.astype(str)),
        "filas": len(df),
        "fecha_min": fechas.min(),
        "fecha_max": fechas.max(),
        "fechas_invalidas": int(fechas.isna().sum()),
        "fechas_duplicadas": int(fechas.dropna().duplicated().sum()),
        "orden_cronologico": bool(fechas.is_monotonic_increasing),
        "nulos_total": int(df.isna().sum().sum()),
        "filas_periodo": int(en_periodo.sum()),
    })

    if numero % 1000 == 0:
        print(f"Leídos {numero} de {len(archivos)} archivos")

cobertura = pd.DataFrame(filas)
print("Lectura terminada:", len(cobertura), "archivos")

# ----------------------------------------------------------
# 3. Esquema: ¿todos los archivos tienen las mismas columnas y tipos?
# ----------------------------------------------------------
print("\n=== ESQUEMAS ENCONTRADOS ===")
print(cobertura.groupby(["columnas", "tipos"]).size().sort_values(ascending=False))
esquema_esperado = cobertura["columnas"] == "|".join(COLUMNAS_ESPERADAS)
print("\nArchivos con las 7 columnas esperadas y en ese orden:", esquema_esperado.sum())

# ----------------------------------------------------------
# 4. Calidad básica de fechas y valores nulos
# ----------------------------------------------------------
print("\n=== CALIDAD BÁSICA ===")
print("Archivos vacíos:", (cobertura["filas"] == 0).sum())
print("Archivos con fechas inválidas:", (cobertura["fechas_invalidas"] > 0).sum())
print("Archivos con fechas duplicadas:", (cobertura["fechas_duplicadas"] > 0).sum())
print("Archivos fuera de orden cronológico:", (~cobertura["orden_cronologico"]).sum())
print("Archivos con algún valor nulo:", (cobertura["nulos_total"] > 0).sum())

# ----------------------------------------------------------
# 5. Cobertura temporal
# ----------------------------------------------------------
print("\n=== COBERTURA TEMPORAL ===")
print("Primera fecha del dataset:", cobertura["fecha_min"].min())
print("Última fecha del dataset:", cobertura["fecha_max"].max())

print("\nPrimeras fechas más frecuentes:")
print(cobertura["fecha_min"].dt.date.value_counts().head(5))

print("\nTickers según el año de su primera fecha:")
print(cobertura["fecha_min"].dt.year.astype("Int64").value_counts().sort_index())

print("\nFilas por archivo:")
print(cobertura["filas"].describe().round(1))

# Cobertura completa: empieza en septiembre de 2019 y llega al fin del periodo
cobertura["periodo_completo"] = (
    (cobertura["fecha_min"] <= pd.Timestamp("2019-09-30"))
    & (cobertura["fecha_max"] >= FIN_PERIODO)
)
completos = cobertura[cobertura["periodo_completo"]]
max_sesiones = completos["filas_periodo"].max()
huecos = max_sesiones - completos["filas_periodo"]

print("\n=== TICKERS CON EL PERIODO COMPLETO (sep-2019 a ago-2026) ===")
print("Tickers:", len(completos))
print("Máximo de sesiones en el periodo:", max_sesiones)
print("Tickers con menos sesiones que el máximo:", (huecos > 0).sum())
print("\nSesiones que les faltan respecto al máximo:")
print(huecos[huecos > 0].describe().round(1))

# ----------------------------------------------------------
# 6. Consistencia con el archivo de estado (Actividad 2)
# ----------------------------------------------------------
estado = pd.read_csv(CARPETA / "estado_ingesta.csv",
                     dtype={"ticker": str}, keep_default_na=False)
estado["last_date"] = pd.to_datetime(estado["last_date"], errors="coerce")
comparacion = cobertura.merge(estado[["ticker", "last_date"]], on="ticker", how="left")
ambas_vacias = comparacion["fecha_max"].isna() & comparacion["last_date"].isna()
distintas = (comparacion["fecha_max"] != comparacion["last_date"]) & ~ambas_vacias

print("\n=== CONSISTENCIA CON EL ARCHIVO DE ESTADO ===")
print("Tickers cuya última fecha no coincide con last_date:", distintas.sum())
print(comparacion.loc[distintas, ["ticker", "fecha_max", "last_date"]].head(10).to_string(index=False))

# ----------------------------------------------------------
# 7. Guardar la tabla de cobertura en Drive
# ----------------------------------------------------------
cobertura.to_csv(CARPETA / "cobertura_archivos.csv", index=False)
print("\nGuardado: cobertura_archivos.csv -",
      (CARPETA / "cobertura_archivos.csv").stat().st_size, "bytes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Versión de huggingface_hub: 1.33.0


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6818 files:   0%|          | 0/6818 [00:00<?, ?it/s]


Archivos descargados: 6818
Leídos 1000 de 6818 archivos
Leídos 2000 de 6818 archivos
Leídos 3000 de 6818 archivos
Leídos 4000 de 6818 archivos
Leídos 5000 de 6818 archivos
Leídos 6000 de 6818 archivos
Lectura terminada: 6818 archivos

=== ESQUEMAS ENCONTRADOS ===
columnas                                   tipos                                               
date|open|high|low|close|adj_close|volume  object|float64|float64|float64|float64|float64|int64    6818
dtype: int64

Archivos con las 7 columnas esperadas y en ese orden: 6818

=== CALIDAD BÁSICA ===
Archivos vacíos: 0
Archivos con fechas inválidas: 0
Archivos con fechas duplicadas: 0
Archivos fuera de orden cronológico: 0
Archivos con algún valor nulo: 0

=== COBERTURA TEMPORAL ===
Primera fecha del dataset: 2019-09-05 00:00:00
Última fecha del dataset: 2026-11-03 00:00:00

Primeras fechas más frecuentes:
fecha_min
2019-11-04    4018
2021-01-08      22
2021-06-30      13
2026-11-03      11
2021-03-26      10
Name: count, dtype: i

In [4]:
# ==========================================================
# ACTIVIDAD 4. Cobertura frente al calendario oficial de la bolsa
# Usa la tabla de la Actividad 3. No descarga precios.
# ==========================================================
!pip install -q exchange_calendars

from pathlib import Path

import exchange_calendars as xcals
import pandas as pd
from google.colab import drive

INICIO_AJUSTADO = pd.Timestamp("2019-11-04")   # propuesta: inicio común real
FIN_PERIODO = pd.Timestamp("2026-08-31")

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# ----------------------------------------------------------
# 1. Cargar la tabla de cobertura y el calendario de la NYSE
# ----------------------------------------------------------
cobertura = pd.read_csv(
    CARPETA / "cobertura_archivos.csv",
    dtype={"ticker": str},
    keep_default_na=False,
    parse_dates=["fecha_min", "fecha_max"],
)
calendario = xcals.get_calendar("XNYS", start="2019-01-01", end="2026-12-31")
sesiones = calendario.sessions        # días en que la bolsa abrió

print("=== CONTROL DE CARGA ===")
print("Tickers:", len(cobertura))
print("Sesiones oficiales en el calendario:", len(sesiones),
      "| de", sesiones.min().date(), "a", sesiones.max().date())

# ----------------------------------------------------------
# 2. ¿El repositorio guarda una ventana móvil de 7 años?
# ----------------------------------------------------------
cobertura["anios_historial"] = (
    (cobertura["fecha_max"] - cobertura["fecha_min"]).dt.days / 365.25
).round(3)
empiezan_2019 = cobertura[cobertura["fecha_min"].dt.year == 2019]

print("\n=== VENTANA DE 7 AÑOS (tickers que empiezan en 2019) ===")
print("Tickers:", len(empiezan_2019))
print("\nAños de historial:")
print(empiezan_2019["anios_historial"].describe().round(3))
print("\nPares (primera fecha, última fecha) más frecuentes:")
print(empiezan_2019.groupby(["fecha_min", "fecha_max"]).size()
      .sort_values(ascending=False).head(8))

# ----------------------------------------------------------
# 3. Huecos exactos: filas del archivo frente a sesiones oficiales
# ----------------------------------------------------------
def sesiones_entre(inicio, fin):
    """Cuenta las sesiones oficiales entre dos fechas, ambas incluidas."""
    return int(((sesiones >= inicio) & (sesiones <= fin)).sum())

cobertura["sesiones_esperadas"] = cobertura.apply(
    lambda fila: sesiones_entre(fila["fecha_min"], fila["fecha_max"]), axis=1
)
cobertura["diferencia"] = cobertura["filas"] - cobertura["sesiones_esperadas"]

print("\n=== FILAS FRENTE A SESIONES OFICIALES ===")
print("Sin diferencia:", (cobertura["diferencia"] == 0).sum())
print("Les faltan sesiones (diferencia < 0):", (cobertura["diferencia"] < 0).sum())
print("Tienen filas de más (diferencia > 0):", (cobertura["diferencia"] > 0).sum())
print("\nDistribución de las diferencias distintas de cero:")
print(cobertura.loc[cobertura["diferencia"] != 0, "diferencia"].value_counts().sort_index())

# ----------------------------------------------------------
# 4. Universo que cubre el periodo ajustado completo
# ----------------------------------------------------------
cobertura["cubre_periodo_ajustado"] = (
    (cobertura["fecha_min"] <= INICIO_AJUSTADO)
    & (cobertura["fecha_max"] >= FIN_PERIODO)
)
cubren = cobertura["cubre_periodo_ajustado"]

print("\n=== PERIODO AJUSTADO (4-nov-2019 a 31-ago-2026) ===")
print("Sesiones oficiales en el periodo:", sesiones_entre(INICIO_AJUSTADO, FIN_PERIODO))
print("Tickers que lo cubren completo:", cubren.sum())
print("  de ellos, sin diferencia frente al calendario:",
      (cubren & (cobertura["diferencia"] == 0)).sum())
print("Tickers que empiezan después del 4-nov-2019:",
      (cobertura["fecha_min"] > INICIO_AJUSTADO).sum())
print("Tickers que terminan antes del 31-ago-2026:",
      (cobertura["fecha_max"] < FIN_PERIODO).sum())

# ----------------------------------------------------------
# 5. Guardar la tabla ampliada (la de la Actividad 3 no se modifica)
# ----------------------------------------------------------
cobertura.to_csv(CARPETA / "cobertura_calendario.csv", index=False)
print("\nGuardado: cobertura_calendario.csv -",
      (CARPETA / "cobertura_calendario.csv").stat().st_size, "bytes")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== CONTROL DE CARGA ===
Tickers: 6818
Sesiones oficiales en el calendario: 2011 | de 2019-01-02 a 2026-12-31

=== VENTANA DE 7 AÑOS (tickers que empiezan en 2019) ===
Tickers: 4099

Años de historial:
count    4099.000
mean        6.997
std         0.007
min         6.842
25%         6.998
50%         6.998
75%         6.998
max         6.998
Name: anios_historial, dtype: float64

Pares (primera fecha, última fecha) más frecuentes:
fecha_min   fecha_max 
2019-11-04  2026-11-03    3879
            2026-11-02     136
2019-09-26  2026-09-25       5
2019-11-05  2026-11-03       4
2019-10-07  2026-1

In [5]:
# ==========================================================
# ACTIVIDAD 5. Explorar el directorio de Nasdaq Trader y
# emparejarlo con nuestros tickers. Todavía no clasificamos.
# ==========================================================
from pathlib import Path

import pandas as pd
import requests
from google.colab import drive

URL_BASE = "https://www.nasdaqtrader.com/dynamic/SymDir/"
ARCHIVOS = ["nasdaqlisted.txt", "otherlisted.txt"]
BOLSAS = {"A": "NYSE American", "N": "NYSE", "P": "NYSE Arca",
          "Z": "Cboe BZX", "V": "IEX"}

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
CARPETA_NASDAQ = CARPETA / "nasdaq_trader"     # copia personal, no se publica
CARPETA_NASDAQ.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

# ----------------------------------------------------------
# 1. Descargar los dos archivos y guardar una copia sin modificar
# ----------------------------------------------------------
fecha = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d")
rutas = {}
for nombre in ARCHIVOS:
    respuesta = requests.get(URL_BASE + nombre, timeout=60,
                             headers={"User-Agent": "Mozilla/5.0"})
    respuesta.raise_for_status()                 # se detiene si la descarga falla
    ruta = CARPETA_NASDAQ / f"{fecha}_{nombre}"
    ruta.write_bytes(respuesta.content)
    rutas[nombre] = ruta
    print("Descargado:", ruta.name, "-", len(respuesta.content), "bytes")

# ----------------------------------------------------------
# 2. Leer los archivos: separados por "|" y con una fila final de control
# ----------------------------------------------------------
def leer_directorio(ruta):
    tabla = pd.read_csv(ruta, sep="|", dtype=str, keep_default_na=False)
    es_pie = tabla.iloc[:, 0].str.startswith("File Creation Time")
    print(f"{ruta.name}: {len(tabla) - es_pie.sum()} símbolos |",
          tabla.loc[es_pie].iloc[0, 0] if es_pie.any() else "sin fila de control")
    return tabla.loc[~es_pie]

print("\n=== CONTENIDO ===")
nasdaq = leer_directorio(rutas["nasdaqlisted.txt"])
otras = leer_directorio(rutas["otherlisted.txt"])
print("\nColumnas de nasdaqlisted:", list(nasdaq.columns))
print("Columnas de otherlisted:", list(otras.columns))

# ----------------------------------------------------------
# 3. Unificar ambos archivos con el formato de símbolo de Yahoo
# ----------------------------------------------------------
# Nasdaq usa el mismo símbolo que Yahoo. En las demás bolsas Yahoo escribe
# "BF.B" como "BF-B" y la preferente "ABR$D" como "ABR-PD".
directorio = pd.concat([
    pd.DataFrame({
        "ticker": nasdaq["Symbol"],
        "simbolo_original": nasdaq["Symbol"],
        "nombre_valor": nasdaq["Security Name"],
        "bolsa": "Nasdaq",
        "etf": nasdaq["ETF"],
        "prueba": nasdaq["Test Issue"],
    }),
    pd.DataFrame({
        "ticker": otras["ACT Symbol"].str.replace("$", "-P", regex=False)
                                     .str.replace(".", "-", regex=False),
        "simbolo_original": otras["ACT Symbol"],
        "nombre_valor": otras["Security Name"],
        "bolsa": otras["Exchange"].map(BOLSAS).fillna("Otra"),
        "etf": otras["ETF"],
        "prueba": otras["Test Issue"],
    }),
], ignore_index=True)

print("\n=== DIRECTORIO UNIFICADO ===")
print("Símbolos:", len(directorio), "| repetidos:", directorio["ticker"].duplicated().sum())
print(pd.crosstab(directorio["bolsa"], directorio["etf"], margins=True, margins_name="Total"))
print("Símbolos de prueba:", (directorio["prueba"] == "Y").sum())

# ----------------------------------------------------------
# 4. Emparejar con nuestros tickers
# ----------------------------------------------------------
cobertura = pd.read_csv(CARPETA / "cobertura_calendario.csv",
                        dtype={"ticker": str}, keep_default_na=False)
cobertura["candidato"] = cobertura["cubre_periodo_ajustado"] & (cobertura["diferencia"] == 0)

emparejado = cobertura.merge(directorio.drop_duplicates("ticker"),
                             on="ticker", how="left")
emparejado["en_directorio"] = emparejado["nombre_valor"].notna()
candidatos = emparejado[emparejado["candidato"]]

print("\n=== EMPAREJAMIENTO ===")
print("Nuestros tickers encontrados en el directorio:",
      emparejado["en_directorio"].sum(), "de", len(emparejado))
print("Candidatos (cubren el periodo y sin huecos):", len(candidatos))
print("  encontrados en el directorio:", candidatos["en_directorio"].sum())

print("\nCandidatos encontrados, por bolsa y ETF:")
print(pd.crosstab(candidatos["bolsa"], candidatos["etf"], margins=True, margins_name="Total"))
print("Candidatos marcados como símbolo de prueba:", (candidatos["prueba"] == "Y").sum())

print("\nCandidatos NO encontrados (máx. 40):")
print(sorted(candidatos.loc[~candidatos["en_directorio"], "ticker"])[:40])

print("\n¿Está MTEST en el directorio?")
print(emparejado.loc[emparejado["ticker"] == "MTEST",
                     ["ticker", "en_directorio", "nombre_valor", "prueba"]].to_string(index=False))

# ----------------------------------------------------------
# 5. Primer vistazo al tipo de valor (no es todavía una clasificación)
# ----------------------------------------------------------
patrones = {
    "Common Stock": r"common stock",
    "Ordinary Shares": r"ordinary share",
    "Depositary Shares / ADS": r"depositary|depository|\bADS\b|\bADR\b",
    "Preferred": r"preferred",
    "Units": r"\bunits?\b",
    "Warrants": r"warrant",
    "Rights": r"\brights?\b",
    "Acquisition Corp (posible SPAC)": r"acquisition corp",
    "Fund / Trust / ETF en el nombre": r"\bfund\b|\btrust\b|\betf\b",
}
nombres = candidatos.loc[candidatos["en_directorio"] & (candidatos["etf"] == "N"), "nombre_valor"]
print(f"\n=== PALABRAS CLAVE EN LOS NOMBRES ({len(nombres)} candidatos no ETF) ===")
print("Un mismo nombre puede contener varias palabras clave.")
for etiqueta, patron in patrones.items():
    print(f"{etiqueta:35s} {nombres.str.contains(patron, case=False, regex=True).sum():6d}")

# ----------------------------------------------------------
# 6. Guardar la tabla de emparejamiento (uso interno)
# ----------------------------------------------------------
emparejado.to_csv(CARPETA / "emparejamiento_nasdaq.csv", index=False)
print("\nGuardado: emparejamiento_nasdaq.csv -",
      (CARPETA / "emparejamiento_nasdaq.csv").stat().st_size, "bytes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Descargado: 20261119_nasdaqlisted.txt - 348493 bytes
Descargado: 20261119_otherlisted.txt - 544575 bytes

=== CONTENIDO ===
20261119_nasdaqlisted.txt: 5610 símbolos | File Creation Time: 1119202612:11
20261119_otherlisted.txt: 7692 símbolos | File Creation Time: 1119202612:11

Columnas de nasdaqlisted: ['Symbol', 'Security Name', 'Market Category', 'Test Issue', 'Financial Status', 'Round Lot Size', 'ETF', 'NextShares']
Columnas de otherlisted: ['ACT Symbol', 'Security Name', 'Exchange', 'CQS Symbol', 'ETF', 'Round Lot Size', 'Test Issue', 'NASDAQ Symbol']

=== DIRECTORIO UNIFICADO ===
Símbolos: 13302 | repetidos: 0
etf               N     Y  Total
bolsa                           
Cboe BZX          6  1669   1675
IEX               3     0      3
NYSE           2810    81   2891
NYSE American   313     0    313
NYSE Arca        20  2760   2780
Nasdaq         4

In [6]:
# ==========================================================
# ACTIVIDAD 6. Clasificación preliminar por tipo de valor
# Asigna una clase a cada candidato. Todavía no excluye nada.
# ==========================================================
import re
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 95)

# ----------------------------------------------------------
# 1. Cargar el emparejamiento de la Actividad 5
# ----------------------------------------------------------
datos = pd.read_csv(CARPETA / "emparejamiento_nasdaq.csv",
                    dtype=str, keep_default_na=False)
for columna in ["candidato", "en_directorio"]:
    datos[columna] = datos[columna] == "True"
datos["fecha_max"] = pd.to_datetime(datos["fecha_max"])

candidatos = datos[datos["candidato"]].copy()
print("Candidatos:", len(candidatos))

# ----------------------------------------------------------
# 2. Reglas de clasificación: se aplica la PRIMERA que coincida
# ----------------------------------------------------------
# El orden importa. Por ejemplo, "Depositary Shares ... Preferred Stock"
# debe ser preferente y no ADR, por eso "preferente" va antes que "adr".
REGLAS = [
    ("deuda",            r"\bnotes?\b|debenture|\bbonds?\b"),
    ("warrant",          r"warrant"),
    ("derecho",          r"\brights?\b"),
    ("preferente",       r"preferred|preference"),
    ("mlp",              r"common units|limited partner"),
    ("unidad",           r"\bunits?\b"),
    ("adr",              r"depositary|depository|\bads\b|\badr\b"),
    ("fondo",            r"\bfund\b|closed[- ]end"),
    ("revisar_trust",    r"\btrust\b|beneficial interest"),
    ("accion_ordinaria", r"common stock|common shares?|ordinary shares?|capital stock"),
]

def clasificar(fila):
    if not fila["en_directorio"]:
        return "sin_directorio"
    if fila["etf"] == "Y":
        return "etf"
    if fila["prueba"] == "Y":
        return "prueba"
    nombre = fila["nombre_valor"].lower()
    for clase, patron in REGLAS:
        if re.search(patron, nombre):
            return clase
    return "sin_clasificar"

candidatos["clase"] = candidatos.apply(clasificar, axis=1)

# ----------------------------------------------------------
# 3. Resultado de la clasificación
# ----------------------------------------------------------
print("\n=== CANDIDATOS POR CLASE ===")
print(candidatos["clase"].value_counts())

print("\n=== EJEMPLOS ALEATORIOS POR CLASE (máx. 8) ===")
for clase in candidatos["clase"].value_counts().index:
    if clase in ("sin_directorio", "revisar_trust", "sin_clasificar"):
        continue                                   # se muestran completos más abajo
    grupo = candidatos[candidatos["clase"] == clase]
    ejemplos = grupo.sample(n=min(8, len(grupo)), random_state=42)
    print(f"\n--- {clase} ({len(grupo)}) ---")
    print(ejemplos[["ticker", "bolsa", "nombre_valor"]].to_string(index=False))

for clase in ["sin_clasificar", "revisar_trust"]:
    grupo = candidatos[candidatos["clase"] == clase].sort_values("ticker")
    print(f"\n=== {clase.upper()} ({len(grupo)}): lista completa, máx. 150 ===")
    print(grupo[["ticker", "bolsa", "nombre_valor"]].head(150).to_string(index=False))

# ----------------------------------------------------------
# 4. Candidatos que no están en el directorio: ¿cuándo terminan?
# ----------------------------------------------------------
sin_dir = candidatos[candidatos["clase"] == "sin_directorio"]
print(f"\n=== SIN DIRECTORIO ({len(sin_dir)}) ===")
print("Terminan antes del 3-nov-2026:", (sin_dir["fecha_max"] < "2026-11-03").sum())
print("Última fecha, por mes:")
print(sin_dir["fecha_max"].dt.to_period("M").value_counts().sort_index())

# ----------------------------------------------------------
# 5. Guardar la clasificación preliminar (uso interno)
# ----------------------------------------------------------
columnas = ["ticker", "clase", "bolsa", "nombre_valor", "fecha_min", "fecha_max", "filas"]
candidatos[columnas].to_csv(CARPETA / "clasificacion_preliminar.csv", index=False)
print("\nGuardado: clasificacion_preliminar.csv -",
      (CARPETA / "clasificacion_preliminar.csv").stat().st_size, "bytes")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Candidatos: 4046

=== CANDIDATOS POR CLASE ===
clase
accion_ordinaria    3073
fondo                237
adr                  209
preferente           179
revisar_trust        131
sin_clasificar        84
sin_directorio        67
mlp                   34
deuda                 18
unidad                 8
etf                    5
derecho                1
Name: count, dtype: int64

=== EJEMPLOS ALEATORIOS POR CLASE (máx. 8) ===

--- accion_ordinaria (3073) ---
ticker  bolsa                                    nombre_valor
   TKC   NYSE    Turkcell Iletisim Hizmetleri AS Common Stock
    MG   NYSE                  Mistras Group Inc Common Stock
  SNAP   NYSE                  Snap Inc. Class A Common Stock
   AQN   NYSE Algonquin Power & Utilities Corp. Common Shares
  ATER Nasdaq                    Aterian, Inc. - Common Stock
   TXN Nasdaq   Texas Instruments Incor

In [7]:
# ==========================================================
# ACTIVIDAD 7. Explorar los datos de la SEC (EDGAR)
# Tabla de tickers + prueba piloto con unas 20 empresas.
# ==========================================================
import time
from pathlib import Path

import pandas as pd
import requests
from google.colab import drive

# La SEC exige identificarse con un correo de contacto en cada consulta
CORREO = "tu_correo@ejemplo.com"   # <-- escribe aquí tu correo de contacto (lo exige la SEC)
assert "@" in CORREO and "ejemplo" not in CORREO, "Escribe tu correo en la variable CORREO"
CABECERAS = {"User-Agent": f"TFM EBIS {CORREO}", "Accept-Encoding": "gzip, deflate"}
PAUSA = 0.2        # 5 consultas por segundo, por debajo del límite de 10 de la SEC

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
CARPETA_SEC = CARPETA / "sec_edgar"
CARPETA_SEC.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 45)

# ----------------------------------------------------------
# 1. Descargar la tabla oficial ticker -> CIK y guardar una copia
# ----------------------------------------------------------
respuesta = requests.get("https://www.sec.gov/files/company_tickers_exchange.json",
                         headers=CABECERAS, timeout=60)
respuesta.raise_for_status()
fecha = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d")
(CARPETA_SEC / f"{fecha}_company_tickers_exchange.json").write_bytes(respuesta.content)

contenido = respuesta.json()
print("=== TABLA DE TICKERS DE LA SEC ===")
print("Claves del archivo:", list(contenido.keys()))
columnas = contenido.get("fields", ["cik", "name", "ticker", "exchange"])
sec = pd.DataFrame(contenido["data"], columns=columnas)
sec["cik"] = sec["cik"].astype(int).astype(str).str.zfill(10)   # CIK de 10 dígitos
sec["ticker"] = sec["ticker"].astype(str).str.upper()

print("Registros:", len(sec), "| CIK únicos:", sec["cik"].nunique(),
      "| tickers repetidos:", sec["ticker"].duplicated().sum())
print(sec["exchange"].value_counts(dropna=False))
print(sec.head(5).to_string(index=False))

# ----------------------------------------------------------
# 2. Emparejar nuestros candidatos con la SEC
# ----------------------------------------------------------
clasif = pd.read_csv(CARPETA / "clasificacion_preliminar.csv",
                     dtype=str, keep_default_na=False)
emp = clasif.merge(
    sec.drop_duplicates("ticker")[["ticker", "cik", "name", "exchange"]],
    on="ticker", how="left",
)
emp["en_sec"] = emp["cik"].notna()

print("\n=== EMPAREJAMIENTO POR CLASE PRELIMINAR ===")
print(pd.crosstab(emp["clase"], emp["en_sec"], margins=True, margins_name="Total"))
print("CIK únicos entre los candidatos encontrados:", emp.loc[emp["en_sec"], "cik"].nunique())

acciones = emp[emp["clase"] == "accion_ordinaria"]
print("\nAcciones ordinarias NO encontradas en la SEC (máx. 40):")
print(sorted(acciones.loc[~acciones["en_sec"], "ticker"])[:40])

varias = (acciones[acciones["en_sec"]].groupby("cik")["ticker"]
          .agg(lambda t: ", ".join(sorted(t))))
varias = varias[varias.str.contains(",")]
print(f"\nCIK con varios tickers de acción ordinaria: {len(varias)} (máx. 15)")
print(varias.head(15).to_string())

# ----------------------------------------------------------
# 3. Prueba piloto: historial de informes de unas 20 empresas
# ----------------------------------------------------------
PILOTO = ["AAPL", "V", "O", "GOOGL", "GOOG", "BF-A", "BF-B",   # nacionales y clases
          "ACN", "MDT", "CB", "GRMN",                           # domicilio extranjero
          "HSBC", "TSM", "NVS", "TKC", "CNQ", "SHOP",           # posibles emisores extranjeros
          "MAIN", "GAM", "EPD", "MSB"]                          # BDC, fondo, L.P., fideicomiso
ANUALES = ["10-K", "10-K/A", "20-F", "20-F/A", "40-F", "40-F/A", "N-CSR", "N-CSRS"]

piloto = emp[emp["ticker"].isin(PILOTO) & emp["en_sec"]]
print("\n=== PRUEBA PILOTO ===")
print("Tickers del piloto encontrados:", len(piloto), "| CIK a consultar:", piloto["cik"].nunique())
faltan = sorted(set(PILOTO) - set(piloto["ticker"]))
print("No están entre los candidatos o no están en la SEC:", faltan)

resultados = []
for numero, cik in enumerate(piloto["cik"].unique()):
    r = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json",
                     headers=CABECERAS, timeout=60)
    time.sleep(PAUSA)
    if r.status_code != 200:
        resultados.append({"cik": cik, "error": r.status_code})
        continue
    datos = r.json()
    if numero == 0:
        print("Campos de la respuesta:", sorted(datos.keys()))

    recientes = pd.DataFrame(datos["filings"]["recent"])[["form", "filingDate"]]
    recientes = recientes.sort_values("filingDate", ascending=False)
    anuales = recientes[recientes["form"].isin(ANUALES)]
    resultados.append({
        "cik": cik,
        "tipo_entidad": datos.get("entityType"),
        "sic": datos.get("sic"),
        "sector_sic": datos.get("sicDescription"),
        "constitucion": datos.get("stateOfIncorporation"),
        "ultimo_anual": anuales["form"].iloc[0] if len(anuales) else "ninguno",
        "fecha_anual": anuales["filingDate"].iloc[0] if len(anuales) else "",
        "informes_recientes": len(recientes),
        "desde": recientes["filingDate"].min(),
    })

tabla = piloto[["ticker", "clase", "cik"]].merge(pd.DataFrame(resultados), on="cik", how="left")
print(tabla.sort_values("ticker").to_string(index=False))

# ----------------------------------------------------------
# 4. Guardar el emparejamiento y el piloto (uso interno)
# ----------------------------------------------------------
emp.to_csv(CARPETA / "emparejamiento_sec.csv", index=False)
tabla.to_csv(CARPETA_SEC / "piloto_submissions.csv", index=False)
print("\nGuardados: emparejamiento_sec.csv y sec_edgar/piloto_submissions.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== TABLA DE TICKERS DE LA SEC ===
Claves del archivo: ['data', 'fields']
Registros: 10398 | CIK únicos: 7993 | tickers repetidos: 0
exchange
Nasdaq    4354
NYSE      3255
OTC       2572
None       158
CBOE        44
TXSE        15
Name: count, dtype: int64
       cik           name ticker exchange
0000320193     Apple Inc.   AAPL   Nasdaq
0000789019 MICROSOFT CORP   MSFT   Nasdaq
0001045810    NVIDIA CORP   NVDA   Nasdaq
0001652044  Alphabet Inc.  GOOGL   Nasdaq
0001018724 AMAZON COM INC   AMZN   Nasdaq

=== EMPAREJAMIENTO POR CLASE PRELIMINAR ===
en_sec            False  True  Total
clase                               
accion_ordinaria      6  3067   3073
adr                   0   209    209
derecho               0     1      1
deuda                 0    18     18
etf                   0     5      5
fondo                 0   237    237
mlp                 